[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_48_OTel_Tracing.ipynb)

# Lesson 48 -- OTel Tracing: See Inside Every AI Pipeline Call

**Track 5 - Production Operations** | Lesson 3 of 5

---

## Track 5 Roadmap

| Lesson | Topic | Status |
|--------|-------|--------|
| L46 | Durable Execution: Crash-Safe AI Pipelines | Done |
| L47 | GPU Autoscaling: vLLM Dynamic Batching + Kubernetes HPA | Done |
| **L48** | **OTel Tracing: Per-span breakdown of AI pipelines** | You are here |
| L49 | Eval at Scale: CI Harness + Regression Gates | Next |
| L50 | Track 5 Capstone: Production-Ready Agent Service | Coming |

---

## Why This Lesson Matters

You have built multi-agent pipelines that call vLLM, hit A2A agents, use tools, and run durable workflows.  
When something is **slow** or **wrong in production**, you need to know *exactly* where -- not guess.

**Logging tells you what happened. Tracing tells you why it was slow.**

A trace is a single end-to-end request broken into **spans** -- one per operation:

```
[Orchestrator.run              -------------------------------- 3.2s ]
  [searcher.a2a_call          ----------- 1.8s ]
    [vllm.generate            ------ 1.4s ]
    [tool.web_search          --- 0.3s ]
  [synthesizer.a2a_call       ------- 0.9s ]
    [vllm.generate            ------ 0.7s ]
  [critic.a2a_call            ---- 0.4s ]
    [vllm.generate            --- 0.2s ]
```

**Without tracing:** The request took 3.2 seconds.  
**With tracing:** The searcher's vLLM call dominated at 1.4s -- optimize it first.

---

## OpenTelemetry (OTel) Core Concepts

```
TRACE = one end-to-end request (has a global trace_id)
  +--> SPAN = one operation within the trace
         +-- span_id + parent_span_id
         +-- start_time, end_time (-> duration)
         +-- attributes: key-value pairs (model, tokens, cost)
         +-- events: timestamped log points within a span
         +-- status: OK | ERROR
```

**Context propagation:** Orchestrator injects a `traceparent` header on outgoing HTTP calls.  
The receiving agent extracts it and creates a child span -- linking two processes into one trace.

```
W3C traceparent format:
00 - {32-hex trace_id} - {16-hex parent_span_id} - {flags}
 ^         ^                      ^                  ^
ver   same across ALL         caller's span_id   01=sampled
      processes
```

**Four OTel signal types:**

| Signal | What it answers | Example |
|--------|----------------|----------|
| Traces | Where did time go? | LLM call took 1.4s |
| Metrics | How much/how often? | 45 req/s, p95=2.1s |
| Logs | What happened (unstructured)? | Circuit breaker opened |
| Events | What happened (within a span)? | token_limit_hit |

We focus on **traces** -- the hardest to retrofit but highest signal-to-noise for AI pipelines.


In [ ]:
# Setup
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc anthropic nest_asyncio matplotlib pandas -q

import os, time, uuid, json, asyncio, threading
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

import nest_asyncio
nest_asyncio.apply()

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-PASTE_YOUR_KEY_HERE'
    print('Set ANTHROPIC_API_KEY via Colab Secrets (key icon in left sidebar)')

import anthropic
client = anthropic.Anthropic()
HAIKU  = 'claude-haiku-4-5'
SONNET = 'claude-sonnet-4-6'
print(f'SDK ready | haiku={HAIKU}')


In [ ]:
# OTel Tracer Setup (InMemory exporter for Colab)
from opentelemetry import trace, context as otel_context
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator
from opentelemetry.trace import StatusCode, SpanKind

_exporter = InMemorySpanExporter()

provider = TracerProvider(
    resource=Resource.create({'service.name': 'auto-researcher', 'service.version': '1.0.0'})
)
provider.add_span_processor(SimpleSpanProcessor(_exporter))
trace.set_tracer_provider(provider)

tracer    = trace.get_tracer('auto_researcher', '1.0.0')
propagator = TraceContextTextMapPropagator()

def reset_exporter():       _exporter.clear()
def get_finished_spans():   return _exporter.get_finished_spans()

print('OTel tracer ready (InMemorySpanExporter -- no Jaeger server needed)')
print('  tracer            -> create spans')
print('  propagator        -> inject/extract traceparent headers')
print('  get_finished_spans() -> inspect recorded spans')


## Section 1 -- Instrumenting a Single LLM Call

The most important span in any AI pipeline is the **LLM call span**.  
Use the emerging `gen_ai.*` OTel semantic conventions:

| Attribute | Example value | Why |
|-----------|--------------|-----|
| `gen_ai.system` | `anthropic` | Which provider |
| `gen_ai.request.model` | `claude-haiku-4-5` | Which model |
| `gen_ai.request.max_tokens` | `512` | Config |
| `gen_ai.usage.input_tokens` | `143` | Cost |
| `gen_ai.usage.output_tokens` | `87` | Cost |
| `llm.cost_usd` | `0.000147` | Budget tracking |
| `llm.tag` | `searcher` | Which agent called it |
| `http.status_code` | `200` | Did it succeed? |

> `gen_ai.*` is the 2024 draft OTel spec adopted by LangChain, LlamaIndex, and OpenLLMetry.


In [ ]:
# LLM Call Instrumentation Helper

PRICES = {
    'claude-haiku-4-5':  {'input': 0.80,  'output': 4.00},
    'claude-sonnet-4-6': {'input': 3.00,  'output': 15.00},
}

def cost_of(model, in_tok, out_tok):
    p = PRICES.get(model, {'input': 3.0, 'output': 15.0})
    return (in_tok * p['input'] + out_tok * p['output']) / 1_000_000

def traced_llm_call(system, user, model=HAIKU, max_tokens=512, tag='default', parent_context=None):
    ctx = parent_context or otel_context.get_current()
    with tracer.start_as_current_span(f'llm.call.{tag}', context=ctx, kind=SpanKind.CLIENT) as span:
        # Set request attributes BEFORE the call
        span.set_attribute('gen_ai.system', 'anthropic')
        span.set_attribute('gen_ai.request.model', model)
        span.set_attribute('gen_ai.request.max_tokens', max_tokens)
        span.set_attribute('llm.tag', tag)
        span.set_attribute('llm.prompt_chars', len(system) + len(user))
        t0 = time.time()
        try:
            resp = client.messages.create(
                model=model, max_tokens=max_tokens, system=system,
                messages=[{'role': 'user', 'content': user}],
            )
            latency_ms = (time.time() - t0) * 1000
            in_tok, out_tok = resp.usage.input_tokens, resp.usage.output_tokens
            cost = cost_of(model, in_tok, out_tok)
            text = resp.content[0].text
            # Set response attributes AFTER the call (still inside the 'with' block!)
            span.set_attribute('gen_ai.usage.input_tokens', in_tok)
            span.set_attribute('gen_ai.usage.output_tokens', out_tok)
            span.set_attribute('llm.cost_usd', round(cost, 6))
            span.set_attribute('llm.latency_ms', round(latency_ms, 1))
            span.set_attribute('http.status_code', 200)
            span.set_status(StatusCode.OK)
            span.add_event('llm.response_received', {'tokens_generated': out_tok, 'stop_reason': resp.stop_reason})
            return {'text': text, 'input_tokens': in_tok, 'output_tokens': out_tok, 'cost_usd': cost}
        except Exception as e:
            span.set_status(StatusCode.ERROR, str(e))
            span.record_exception(e)
            raise

# Demo: trace a single call
reset_exporter()
with tracer.start_as_current_span('demo.single_llm_call'):
    result = traced_llm_call(
        system='You are a concise AI assistant.',
        user='What is OpenTelemetry in one sentence?',
        model=HAIKU, tag='lesson_demo',
    )

print(f'Response: {result["text"][:120]}')
print(f'Tokens: {result["input_tokens"]} in / {result["output_tokens"]} out | Cost: ${result["cost_usd"]:.6f}')
print(f'\nSpans recorded: {len(get_finished_spans())}')
for s in get_finished_spans():
    dur_ms = (s.end_time - s.start_time) / 1_000_000
    print(f'  [{s.name}]  {dur_ms:.1f}ms')
    for k, v in (s.attributes or {}).items():
        if k.startswith('gen_ai') or k.startswith('llm'):
            print(f'    {k} = {v}')


## Section 2 -- Nested Spans: Parent-Child Relationships

When an orchestrator calls multiple agents, spans **nest**:

```
trace_id = '4bf9...4736'        <- same for EVERY span in one request

[orchestrator.run]              span_id=A  parent=none   <- root
  [agent.searcher.contribute]   span_id=B  parent=A
    [llm.call.searcher]         span_id=C  parent=B
  [agent.synthesizer.contribute]span_id=D  parent=A
    [llm.call.synthesizer]      span_id=E  parent=D
  [agent.critic.contribute]     span_id=F  parent=A
    [llm.call.critic]           span_id=G  parent=F
```

The **critical path** is the longest chain of parent->child spans.  
Speeding up non-critical-path spans does nothing for wall time.  

**OTel context flows automatically** via Python's `contextvars.ContextVar`.  
Any span opened inside `with tracer.start_as_current_span():` automatically  
becomes a child of whatever span is currently active -- zero extra plumbing.


In [ ]:
# Nested Agent Pipeline -- orchestrator + 3 agents

def simulate_agent_work(agent_name, topic):
    prompts = {
        'searcher':    f'Find 3 key facts about: {topic}. Be brief.',
        'synthesizer': f'Synthesize into 2 bullet points: {topic}',
        'critic':      f'Rate quality 1-5 of research on: {topic}',
    }
    return traced_llm_call(
        system=f'You are the {agent_name} agent. Be concise.',
        user=prompts[agent_name], model=HAIKU, tag=agent_name,
    )

def run_research_pipeline(topic):
    results = {}
    with tracer.start_as_current_span('orchestrator.run') as root:
        root.set_attribute('pipeline.topic', topic)
        for agent in ['searcher', 'synthesizer', 'critic']:
            with tracer.start_as_current_span(f'agent.{agent}.contribute') as asp:
                t0 = time.time()
                results[agent] = simulate_agent_work(agent, topic)
                asp.set_attribute('agent.latency_ms', round((time.time()-t0)*1000, 1))
                asp.set_attribute('agent.cost_usd', round(results[agent]['cost_usd'], 6))
        root.set_attribute('pipeline.total_cost_usd',
                           round(sum(r['cost_usd'] for r in results.values()), 6))
    return results

reset_exporter()
print('Running 3-agent pipeline...\n')
t_start = time.time()
results = run_research_pipeline('transformer attention mechanisms')
print(f'Completed in {time.time()-t_start:.2f}s | Spans: {len(get_finished_spans())}')
print()
for agent, r in results.items():
    print(f'  {agent}: {r["text"][:80].strip()}...')

print('\nSpan hierarchy:')
for s in sorted(get_finished_spans(), key=lambda x: x.start_time):
    dur = (s.end_time - s.start_time) / 1_000_000
    par = f'parent={format(s.parent.span_id,"016x")[:8]}' if s.parent else 'ROOT'
    print(f'  [{s.name}]  {dur:.0f}ms  {par}')

# EXPERIMENT: Replace sequential loop with asyncio.gather() -- compare total time


In [ ]:
# Trace Analysis: Per-Span Breakdown
import pandas as pd

def analyze_trace(spans):
    rows = []
    for s in spans:
        dur_ms = (s.end_time - s.start_time) / 1_000_000
        parent_id = format(s.parent.span_id, '016x')[:8] if s.parent else 'ROOT'
        rows.append({
            'name':      s.name,
            'span_id':   format(s.context.span_id, '016x')[:8],
            'parent_id': parent_id,
            'dur_ms':    round(dur_ms, 1),
            'status':    s.status.status_code.name,
            **{k: v for k, v in (s.attributes or {}).items()
               if k in ['gen_ai.usage.input_tokens', 'gen_ai.usage.output_tokens',
                        'llm.cost_usd', 'llm.tag']},
        })
    return pd.DataFrame(rows).sort_values('dur_ms', ascending=False)

df = analyze_trace(get_finished_spans())
print('=== Span breakdown (sorted by duration) ===')
print(df[['name','span_id','parent_id','dur_ms','status']].to_string(index=False))

llm_spans = [s for s in get_finished_spans() if 'llm.call' in s.name]
total_cost = sum(s.attributes.get('llm.cost_usd', 0) for s in llm_spans)
total_in   = sum(s.attributes.get('gen_ai.usage.input_tokens', 0) for s in llm_spans)
total_out  = sum(s.attributes.get('gen_ai.usage.output_tokens', 0) for s in llm_spans)
print(f'\nLLM spans={len(llm_spans)}  input_tok={total_in}  output_tok={total_out}  cost=${total_cost:.6f}')


In [ ]:
# Flamegraph Visualization (Gantt chart)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_trace_gantt(spans, title='Trace Timeline'):
    if not spans:
        print('No spans to plot.')
        return
    trace_start = min(s.start_time for s in spans)
    rows = []
    for s in spans:
        start_ms = (s.start_time - trace_start) / 1_000_000
        dur_ms   = (s.end_time  - s.start_time) / 1_000_000
        rows.append({'name': s.name, 'start_ms': start_ms, 'dur_ms': dur_ms,
                     'is_llm': 'llm.call' in s.name,
                     'is_agent': s.name.startswith('agent.'),
                     'is_root':  s.parent is None})
    rows.sort(key=lambda r: (not r['is_root'], not r['is_agent'], r['start_ms']))

    colors = {'root': '#4e79a7', 'agent': '#f28e2b', 'llm': '#59a14f', 'other': '#9c755f'}
    def color_for(r):
        if r['is_root']:  return colors['root']
        if r['is_agent']: return colors['agent']
        if r['is_llm']:   return colors['llm']
        return colors['other']

    fig, ax = plt.subplots(figsize=(12, max(4, len(rows) * 0.55)))
    for i, row in enumerate(rows):
        c = color_for(row)
        ax.barh(y=i, left=row['start_ms'], width=max(row['dur_ms'], 5),
                height=0.6, color=c, alpha=0.85, edgecolor='white', linewidth=0.5)
        ax.text(row['start_ms'] + max(row['dur_ms'],5)/2, i,
                f"{row['name']}  ({row['dur_ms']:.0f}ms)",
                ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([str(i+1) for i in range(len(rows))], fontsize=8)
    ax.set_xlabel('Time from trace start (ms)', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.legend(handles=[
        mpatches.Patch(color=colors['root'],  label='Orchestrator (root)'),
        mpatches.Patch(color=colors['agent'], label='Agent spans'),
        mpatches.Patch(color=colors['llm'],   label='LLM call spans'),
    ], loc='lower right', fontsize=8)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    total_dur = max(r['start_ms'] + r['dur_ms'] for r in rows)
    llm_rows  = [r for r in rows if r['is_llm']]
    if llm_rows:
        bottleneck = max(llm_rows, key=lambda r: r['dur_ms'])
        llm_total  = sum(r['dur_ms'] for r in llm_rows)
        print('\n=== Critical path ===')
        print(f'  Total trace : {total_dur:.0f}ms')
        print(f'  LLM total   : {llm_total:.0f}ms ({100*llm_total/total_dur:.0f}% of wall time)')
        print(f'  Bottleneck  : [{bottleneck["name"]}] {bottleneck["dur_ms"]:.0f}ms')
        print(f'  Parallel speedup potential: ~{100*(1-1/len(llm_rows)):.0f}%')

plot_trace_gantt(get_finished_spans(), 'Research Pipeline Trace')


## Section 3 -- Context Propagation Across Process Boundaries

In a real deployment, each A2A agent is a separate process.  
OTel links their spans into one trace via the `traceparent` HTTP header.

```
+-------------------+   HTTP POST /tasks/send    +-------------------+
| Orchestrator      | ------------------------>  | Searcher Agent    |
| (process A)       |  traceparent: 00-{tid}-{}  | (process B)       |
| span: orch.run    | <------------------------  | span: a2a.handle  |
|   +--> span A     |       response              |   parent = A      |
+-------------------+                            +-------------------+
```

**Rule:** trace_id never changes across hops.  
Each hop gets a new span_id; its parent_span_id = caller's span_id.

Here we simulate this by passing a dict (the 'HTTP headers') between functions.


In [ ]:
# Context Propagation: Simulating Cross-Process A2A Tracing

reset_exporter()

def orchestrator_process():
    with tracer.start_as_current_span('orchestrator.dispatch') as span:
        span.set_attribute('agent.target', 'searcher')
        # Inject OTel context into 'HTTP headers'
        outgoing_headers = {}
        propagator.inject(outgoing_headers)
        print(f'[Orchestrator] traceparent={outgoing_headers.get("traceparent", "MISSING")}')
        print(f'[Orchestrator] span_id={format(span.get_span_context().span_id, "016x")[:8]}...')
        response = searcher_process(incoming_headers=outgoing_headers, topic='transformers')
        span.set_attribute('a2a.response_chars', len(response))
        return response

def searcher_process(incoming_headers, topic):
    # Extract context from incoming headers
    parent_ctx = propagator.extract(incoming_headers)
    with tracer.start_as_current_span('searcher.handle_task', context=parent_ctx, kind=SpanKind.SERVER) as span:
        span.set_attribute('a2a.task_topic', topic)
        print(f'[Searcher]     parent_span visible={span.parent is not None}')
        print(f'[Searcher]     span_id={format(span.get_span_context().span_id, "016x")[:8]}...')
        result = traced_llm_call(
            system='You are a researcher.',
            user=f'Three facts about {topic}:',
            model=HAIKU, tag='searcher',
        )
        span.set_attribute('llm.cost_usd', result['cost_usd'])
        return result['text']

response = orchestrator_process()
print(f'\nResponse: {response[:100]}...')

spans = get_finished_spans()
print(f'\nTotal spans: {len(spans)}')
print('Span hierarchy:')
for s in sorted(spans, key=lambda x: x.start_time):
    par = f'parent={format(s.parent.span_id,"016x")[:8]}' if s.parent else 'ROOT'
    print(f'  [{s.name}]  id={format(s.context.span_id,"016x")[:8]}  {par}')

trace_ids = {format(s.context.trace_id, '032x') for s in spans}
print(f'\nAll spans share ONE trace_id: {len(trace_ids)==1}')
print(f'trace_id = {list(trace_ids)[0][:20]}...')


## Section 4 -- Sampling Strategies

**Problem:** At 1,000 req/s, storing every span costs ~$200/month in Grafana Tempo.  
**Solution:** Sample -- only record a fraction of traces.

### Head Sampling (decide at trace start)

```python
from opentelemetry.sdk.trace.sampling import TraceIdRatioBased, ParentBased

# Record 10% of all requests
sampler = ParentBased(root=TraceIdRatioBased(0.10))
# ParentBased ensures: if parent was sampled, child is ALWAYS sampled too
# (prevents orphan spans -- children with no parent in the backend)
```

| Strategy | Record rate | Best for |
|----------|------------|----------|
| `ALWAYS_ON` | 100% | Development |
| `TraceIdRatioBased(0.10)` | 10% | Production <1K req/s |
| `TraceIdRatioBased(0.001)` | 0.1% | >10K req/s |
| Tail sampling | Error-biased | Catching rare bugs |

### Tail Sampling (decide after trace completes)

Head sampling is blind -- it might drop the one trace with an error.  
Tail sampling buffers spans in a collector window, then decides:

```yaml
# otelcol-config.yaml
processors:
  tail_sampling:
    decision_wait: 10s
    policies:
      - name: errors-policy
        type: status_code
        status_code: {status_codes: [ERROR]}   # always keep errors
      - name: slow-policy
        type: latency
        latency: {threshold_ms: 2000}           # always keep slow traces
      - name: sample-otherwise
        type: probabilistic
        probabilistic: {sampling_percentage: 1} # 1% of everything else
```

Rule of thumb: dev = ALWAYS_ON | prod <1K req/s = 10% head | prod >10K = tail in collector


In [ ]:
# Sampling Demo
from opentelemetry.sdk.trace.sampling import TraceIdRatioBased, ParentBased

def run_n_requests(n, sampler_name, sample_rate):
    exp = InMemorySpanExporter()
    prov = TracerProvider(
        resource=Resource.create({'service.name': 'sampling-demo'}),
        sampler=ParentBased(root=TraceIdRatioBased(sample_rate)),
    )
    prov.add_span_processor(SimpleSpanProcessor(exp))
    t = prov.get_tracer('test')
    for i in range(n):
        with t.start_as_current_span(f'req.{i}') as s:
            s.set_attribute('request.id', i)
    recorded = len(exp.get_finished_spans())
    print(f'  {sampler_name:42s}  {n} req -> {recorded:4d} recorded  ({100*recorded/n:.1f}%)')

print('=== Sampling rates (simulated, no real LLM calls) ===')
run_n_requests(1000, 'ALWAYS_ON (development)',          1.0)
run_n_requests(1000, '10% head sampling (production)',    0.10)
run_n_requests(1000, '1% head sampling (high traffic)',   0.01)
run_n_requests(1000, 'ALWAYS_OFF (disabled)',             0.0)

print('\nKey rules:')
print('  1. Always wrap samplers in ParentBased() to prevent orphan spans')
print('  2. Never use ALWAYS_OFF in staging -- you cannot debug what you cannot see')
print('  3. Add error + latency policies on top of probabilistic sampling in prod')

# EXPERIMENT: Run run_n_requests(100, ..., 0.5) five times.
# Notice the recorded count varies -- sampling is probabilistic per trace_id hash.


In [ ]:
# Production Export: OTLP -> Jaeger / Grafana Tempo
# In Colab: InMemorySpanExporter. In production: one-line swap to OTLPSpanExporter.
# Everything else (tracer, spans, attributes) stays IDENTICAL.

PROD_CONFIG = '''
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import BatchSpanProcessor  # NOT SimpleSpanProcessor!
from opentelemetry.sdk.trace.sampling import ParentBased, TraceIdRatioBased

# Option A: Jaeger (self-hosted, free)
exporter = OTLPSpanExporter(endpoint="http://localhost:4317", insecure=True)

# Option B: Grafana Tempo (hosted)
# exporter = OTLPSpanExporter(
#     endpoint="https://tempo.grafana.net:443",
#     headers={"Authorization": "Bearer <token>"},
# )

# Option C: Honeycomb (SaaS, excellent for AI workloads)
# exporter = OTLPSpanExporter(
#     endpoint="https://api.honeycomb.io",
#     headers={"x-honeycomb-team": "<api-key>", "x-honeycomb-dataset": "auto-researcher"},
# )

provider = TracerProvider(
    resource=Resource.create({
        "service.name": "auto-researcher",
        "service.version": "1.0.0",
        "deployment.environment": "production",
    }),
    sampler=ParentBased(root=TraceIdRatioBased(0.10)),
)
provider.add_span_processor(BatchSpanProcessor(
    exporter,
    max_queue_size=2048,
    max_export_batch_size=512,
    schedule_delay_millis=5000,  # buffer 5s, non-blocking
))
trace.set_tracer_provider(provider)
# tracer = trace.get_tracer('auto_researcher')  -- identical to Colab version
'''

print('=== Production OTel config (reference) ===')
print(PROD_CONFIG)

JAEGER_COMPOSE = '''
version: "3.8"
services:
  jaeger:
    image: jaegertracing/all-in-one:latest
    ports:
      - "16686:16686"   # Jaeger UI  ->  http://localhost:16686
      - "4317:4317"     # OTLP gRPC  ->  your app sends spans here
      - "4318:4318"     # OTLP HTTP  ->  alternative
    environment:
      - COLLECTOR_OTLP_ENABLED=true
'''
print('=== docker-compose.yml for local Jaeger ===')
print(JAEGER_COMPOSE)
print('Run:   docker compose up jaeger')
print('Open:  http://localhost:16686')


In [ ]:
# Full 4-Agent Pipeline with Complete Observability

@dataclass
class PipelineStats:
    total_latency_ms: float = 0.0
    total_cost_usd: float = 0.0
    agent_stats: Dict[str, dict] = field(default_factory=dict)
    span_count: int = 0

def run_traced_pipeline(topic, agents=None):
    if agents is None:
        agents = ['searcher', 'synthesizer', 'critic']
    stats = PipelineStats()
    reset_exporter()
    t0_pipe = time.time()
    prompts = {
        'searcher':    f'List 3 facts about: {topic}',
        'synthesizer': f'Synthesize in 2 bullets: {topic}',
        'critic':      f'Rate research quality 1-5 on: {topic}. Why?',
        'verifier':    f'Is this safe to publish? Topic: {topic}',
    }
    with tracer.start_as_current_span('pipeline.research') as root:
        root.set_attribute('pipeline.topic', topic)
        root.set_attribute('pipeline.agent_count', len(agents))
        root.add_event('pipeline.started', {'agents': ','.join(agents)})
        for agent_name in agents:
            with tracer.start_as_current_span(f'agent.{agent_name}') as asp:
                asp.set_attribute('agent.name', agent_name)
                t0_agent = time.time()
                result = traced_llm_call(
                    system=f'You are the {agent_name}. Max 2 sentences.',
                    user=prompts.get(agent_name, f'Analyze: {topic}'),
                    model=HAIKU, tag=agent_name,
                )
                lat = (time.time() - t0_agent) * 1000
                asp.set_attribute('agent.latency_ms', round(lat, 1))
                stats.agent_stats[agent_name] = {'latency_ms': lat, 'cost_usd': result['cost_usd'],
                                                  'preview': result['text'][:60]}
                stats.total_cost_usd += result['cost_usd']
        stats.total_latency_ms = (time.time() - t0_pipe) * 1000
        root.set_attribute('pipeline.total_cost_usd', round(stats.total_cost_usd, 6))
        root.add_event('pipeline.completed', {'cost': str(round(stats.total_cost_usd, 6))})
    stats.span_count = len(get_finished_spans())
    return stats

print('Running 4-agent traced pipeline...\n')
stats = run_traced_pipeline(
    'large language model inference optimization',
    agents=['searcher', 'synthesizer', 'critic', 'verifier']
)
print(f'Total latency : {stats.total_latency_ms:.0f}ms')
print(f'Total cost    : ${stats.total_cost_usd:.6f}')
print(f'Spans recorded: {stats.span_count}')
print()
for agent, s in stats.agent_stats.items():
    pct = 100 * s['latency_ms'] / stats.total_latency_ms
    print(f'  {agent:12s}  {s["latency_ms"]:6.0f}ms ({pct:4.1f}%)  ${s["cost_usd"]:.6f}  | {s["preview"]}...')

print()
plot_trace_gantt(get_finished_spans(), '4-Agent Research Pipeline Trace')


In [ ]:
# 10 OTel Pitfalls for AI Pipelines

pitfalls = [
    ('1',  'Adding span attributes AFTER the span closes',
     'span.set_attribute() outside the with block is a silent no-op.',
     'Set ALL attributes inside the with tracer.start_as_current_span(): block.'),
    ('2',  'Storing full prompts as span attributes',
     'Span attributes have a 64KB total limit. A 10K-token prompt silently truncates.',
     'Store llm.prompt_chars and llm.prompt_hash; log full prompts to a log sink.'),
    ('3',  'Using SimpleSpanProcessor in production',
     'SimpleSpanProcessor blocks the request thread on every export (adds ~5ms at 1K req/s).',
     'Use BatchSpanProcessor in production -- it buffers and exports asynchronously.'),
    ('4',  'Missing ParentBased wrapper on samplers',
     'TraceIdRatioBased(0.10) alone can sample a child without sampling its parent.',
     'Always: ParentBased(root=TraceIdRatioBased(0.10)).'),
    ('5',  'Not propagating traceparent in A2A calls',
     'Each A2A hop starts a fresh trace. You see fragments, not end-to-end traces.',
     'Inject headers before httpx.post(); extract in FastAPI lifespan middleware.'),
    ('6',  'Recording float cost with full precision',
     'Float noise: 0.00014700000000000001 -- clutters dashboards.',
     'round(cost, 6) before span.set_attribute().'),
    ('7',  'Emitting an event per streaming token',
     '100 events/call x 1K req/s = OTel collector overload.',
     'One span per LLM call. Events only at first_token_received and generation_complete.'),
    ('8',  'Setting ALWAYS_OFF in staging',
     'Zero traces = zero debugging ability when staging has issues.',
     'Always ALWAYS_ON in dev/staging. Probabilistic sampling is production-only.'),
    ('9',  'Omitting service.name from Resource',
     'All spans appear as unknown_service in Jaeger. Impossible to filter by service.',
     'Always set service.name and service.version in Resource.create().'),
    ('10', 'Adding tracing as an afterthought across 100 call sites',
     'Retrofitting is painful and inconsistent. Some LLM calls always go untraced.',
     'Instrument at the single chokepoint (reliable_call()). One wrapper = full coverage.'),
]

print('=' * 72)
print('10 OTel Pitfalls for AI Pipelines')
print('=' * 72)
for num, title, problem, fix in pitfalls:
    print(f'\n#{num}: {title}')
    print(f'   Problem : {problem}')
    print(f'   Fix     : {fix}')


## Summary

| Concept | What you built |
|---------|---------------|
| **OTel data model** | Traces -> Spans -> Attributes + Events |
| **LLM span attributes** | `gen_ai.*` semantic conventions (model/tokens/cost) |
| **Nested spans** | Orchestrator -> Agent -> LLM call hierarchy |
| **Context propagation** | `traceparent` header inject/extract across A2A hops |
| **Trace analysis** | Per-span breakdown, critical path identification |
| **Flamegraph** | Gantt timeline showing where wall time goes |
| **Sampling** | Head sampling with ParentBased wrapper; tail sampling config |
| **Production export** | OTLP -> Jaeger / Grafana Tempo / Honeycomb |

---

## The One Rule

> **Instrument at the chokepoint.**

Your L24-L31 reliability spine has one `reliable_call()` every LLM call flows through.  
Wrap that one function with `traced_llm_call()` and you get 100% LLM trace coverage  
with zero per-agent changes. That is the production pattern.

---

## Homework (5 exercises)

1. **Parallel pipeline** -- Rewrite `run_traced_pipeline()` with `asyncio.gather()` for all 4 agents.  
   Plot the Gantt chart. How much wall time did you save vs sequential?

2. **Error span** -- Add a try/except that sets `StatusCode.ERROR` on `anthropic.RateLimitError`.  
   Verify it appears in `analyze_trace()` with status=ERROR.

3. **Wire into AutoResearcher** -- Add `traced_llm_call()` into your L23/L31 `reliable_call()`.  
   Run a research query and render the flamegraph. Where does time actually go?

4. **Sampling stats** -- Run `run_n_requests(10_000, ...)` at rates 0.01, 0.05, 0.10.  
   Plot recorded vs total as a bar chart. What is the variance at each rate?

5. **Jaeger locally** -- Run the docker-compose snippet from Section 4, swap to  
   `OTLPSpanExporter(endpoint='http://localhost:4317', insecure=True)`, and open  
   http://localhost:16686. Find your trace in the Jaeger UI and explore the span waterfall.

---

## Next: L49 -- Eval at Scale

How do you know your pipeline is getting better, not just different?  
L49 covers: hundreds of eval cases in parallel, LLM-as-judge rubrics,  
regression gates that block bad deploys, statistical significance testing,  
CI integration, and dashboards that track quality over time.

*See you tomorrow, Gourav.*
